In [144]:
import pandas as pd
import sqlite3

## create a connection to the database using the library sqlite3


In [145]:
connection = sqlite3.connect('../data/checking-logs.sqlite')

## get the schema of the table test

In [146]:
schema = pd.read_sql('PRAGMA table_info(test);', connection)

In [147]:
schema

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## get only the first 10 rows of the table test to check what the table looks like

In [148]:
first_10_rows = pd.read_sql('SELECT * FROM test LIMIT 10', connection)

In [149]:
first_10_rows

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## find among all the users the minimum value of the delta between the first commit of the user and the deadline of the corresponding lab using only one query

In [150]:
schema = pd.read_sql('PRAGMA table_info(deadlines);', connection)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,labs,TEXT,0,None,0
2,2,deadlines,INTEGER,0,None,0


In [151]:
pd.read_sql('SELECT * FROM deadlines LIMIT 5', connection)

,index,labs,deadlines
0,0,laba04,1587945599
1,1,laba04s,1587945599
2,2,laba05,1588550399
3,4,laba06,1590364799
4,5,laba06s,1590364799


In [152]:
query = '''
SELECT
    test.uid,
    (julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(test.first_commit_ts)) * 24 AS diff_hours
FROM test
JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1'
ORDER BY diff_hours
LIMIT 1;
'''

In [153]:
df_min = pd.read_sql(query, connection)

In [154]:
df_min

,uid,diff_hours
0,user_25,2.867236


## do the same thing, but for the maximum, using only one query, the dataframe name is df_max


In [155]:
query = '''
SELECT
    test.uid,
    (julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(test.first_commit_ts)) * 24 AS avg_hours
FROM test
JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1'

'''

In [156]:
df_max = pd.read_sql(query, connection)

In [157]:
df_max

,uid,avg_hours
0,user_1,6.894594
1,user_1,6.796432
2,user_1,28.744572
3,user_1,175.556592
4,user_1,107.606031
5,user_10,39.585084
6,user_10,39.367888
7,user_10,52.542483
8,user_10,132.341698
9,user_10,112.374396


## do the same thing but for the average, using only one query, this time your dataframe should not include the uid column, and the dataframe name is df_avg

In [158]:
query = '''
SELECT
    AVG((julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(test.first_commit_ts)) * 24) AS avg_diff
FROM test
JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1';
'''

In [159]:
df_avg = pd.read_sql(query, connection)

In [160]:
df_avg

,avg_diff
0,89.687686


## we want to test the hypothesis that the users who visited the newsfeed just a few times have the lower delta between the first commit and the deadline. To do this, you need to calculate the correlation coefficient between the number of pageviews and the difference

In [161]:
query = '''
SELECT
    test.uid AS uid,
    AVG((julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(test.first_commit_ts)) * 24) AS avg_diff,
    COUNT(pageviews.uid) AS pageviews
FROM test
JOIN deadlines ON test.labname = deadlines.labs
LEFT JOIN pageviews ON test.uid = pageviews.uid
WHERE test.labname != 'project1'
GROUP BY test.uid;
'''

In [162]:
views_diff = pd.read_sql(query, connection)

In [163]:
views_diff

,uid,avg_diff,pageviews
0,user_1,65.119644,140
1,user_10,75.242310,445
2,user_14,159.568696,429
3,user_17,62.207513,235
4,user_18,6.367907,9
5,user_19,99.440298,64
6,user_21,96.111041,40
7,user_25,93.474751,895
8,user_28,86.793652,745
9,user_3,105.738041,1585


In [164]:
views_diff.corr(numeric_only=True)

,avg_diff,pageviews
avg_diff,1.000000,0.185042
pageviews,0.185042,1.000000


## close the connection

In [165]:
connection.close()